# LeetCode 3450 — Maximum Students on a Single Bench bằng Python

Link đề: https://leetcode.com/problems/maximum-students-on-a-single-bench/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table (đếm theo nhóm, tập hợp `set`)

Skeleton LeetCode:

```python
class Solution:
    def maxStudentsOnBench(self, students: List[List[int]]) -> int:
        ...
```

**Lưu ý — bài Premium (🔒):** đề trong notebook này được viết theo hiểu biết của mình về bài, nên hãy đối chiếu lại với đề gốc nếu bạn có tài khoản LeetCode Premium. Mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm trên hàng nghìn input ngẫu nhiên). Các ví dụ 1–4 bám theo ví dụ mình nhớ của đề; nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi phần tử `[student_id, bench_id]` cho biết học sinh `student_id` ngồi ở ghế dài (bench) số `bench_id`; hãy tìm **số học sinh khác nhau (unique)** nhiều nhất trên **một** ghế; nếu không có học sinh nào thì trả `0`. Một học sinh có thể xuất hiện **nhiều lần** trên cùng một ghế nhưng chỉ tính **một lần**.
2. Nắm các công cụ Python: **giải nén cặp** `for student_id, bench_id in students`, **tập hợp `set`** (tự loại trùng), **`dict` ánh xạ ghế → tập học sinh** (`defaultdict(set)`), `len`, `max(..., default=0)`, `tuple` để đưa cặp vào `set`, và `itertools.groupby`.
3. Đi từ cách **với mỗi ghế, quét lại cả danh sách** → **`dict` ghế → `set` học sinh** → **loại trùng cặp rồi đếm theo ghế (`Counter`)** → **sắp xếp + `groupby`** → **bảng đánh dấu 2 chiều**.
4. Hiểu mấu chốt: đây là bài **gom nhóm rồi đếm phần tử khác nhau** — `set` lo việc loại trùng, `dict` lo việc chia nhóm theo ghế; câu trả lời là `max` của kích thước các `set` đó.
5. Tránh các lỗi kinh điển: đếm **số dòng** của mỗi ghế (không loại trùng), `max` của danh sách **rỗng** gây lỗi, ánh xạ ngược `học sinh → ghế` (một học sinh ngồi nhiều ghế sẽ bị ghi đè), và trả về **mã ghế** thay vì **số học sinh**.

## 2. Đề bài & ví dụ

Cho mảng hai chiều `students`, trong đó `students[i] = [student_id, bench_id]` nghĩa là học sinh `student_id` đang ngồi ở ghế `bench_id`.

Hãy trả về **số lượng học sinh khác nhau (unique) nhiều nhất** ngồi trên **một ghế duy nhất**. Nếu không có học sinh nào, trả về `0`.

**Ghi chú:** một học sinh có thể xuất hiện **nhiều lần** trên cùng một ghế trong dữ liệu, nhưng chỉ được tính **một lần** cho ghế đó.

**Ví dụ 1**

- Input: `students = [[1,2],[2,2],[3,3],[1,3],[2,3]]`
- Output: `3`
- Giải thích: ghế `2` có hai học sinh khác nhau `{1, 2}`; ghế `3` có ba học sinh khác nhau `{3, 1, 2}`. Nhiều nhất là `3`.

**Ví dụ 2**

- Input: `students = [[1,1],[2,1],[3,1],[4,2],[5,2]]`
- Output: `3`
- Giải thích: ghế `1` có `{1, 2, 3}` (3 học sinh), ghế `2` có `{4, 5}` (2 học sinh). Nhiều nhất là `3`.

**Ví dụ 3**

- Input: `students = [[1,1],[1,1]]`
- Output: `1`
- Giải thích: cùng một học sinh `1` ghi hai lần ở ghế `1` nhưng chỉ tính một lần.

**Ví dụ 4**

- Input: `students = []`
- Output: `0`
- Giải thích: không có học sinh nào.

**Ràng buộc:** `0 <= students.length <= 100`; `students[i] = [student_id, bench_id]`; `1 <= student_id <= 100`; `1 <= bench_id <= 100`.

In [1]:
import random
from collections import Counter, defaultdict
from itertools import groupby
from typing import List

examples = [
    ([[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]], 3),
    ([[1, 1], [2, 1], [3, 1], [4, 2], [5, 2]], 3),
    ([[1, 1], [1, 1]], 1),
    ([], 0),
]
for students, expected in examples:
    print(students, "-> kỳ vọng", expected)

[[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]] -> kỳ vọng 3
[[1, 1], [2, 1], [3, 1], [4, 2], [5, 2]] -> kỳ vọng 3
[[1, 1], [1, 1]] -> kỳ vọng 1
[] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Mảng hai chiều và giải nén cặp trong `for`

`students` là một **list của các list nhỏ**, mỗi list nhỏ có hai phần tử `[student_id, bench_id]`. Khi lặp, ta có thể **giải nén (unpack)** ngay trong `for`: `for student_id, bench_id in students:` — Python tự gán phần tử thứ nhất vào `student_id`, phần tử thứ hai vào `bench_id`. Dễ đọc hơn nhiều so với `pair[0]`, `pair[1]`.

In [2]:
students = [[1, 2], [2, 2], [3, 3]]
for student_id, bench_id in students:
    print(f"học sinh {student_id} ngồi ghế {bench_id}")

pair = students[0]
print(pair, pair[0], pair[1])        # [1, 2] 1 2
print(len(students), len([]))        # 3 0

học sinh 1 ngồi ghế 2
học sinh 2 ngồi ghế 2
học sinh 3 ngồi ghế 3
[1, 2] 1 2
3 0


### 3.2. Tập hợp `set`: tự loại phần tử trùng

`set` là tập hợp **không có phần tử lặp**. `s = set()` tạo tập rỗng (chú ý: `{}` là `dict` rỗng, **không** phải `set`). `s.add(x)` thêm `x` — nếu `x` đã có thì **không thay đổi gì**. `len(s)` là số phần tử khác nhau. Kiểm tra `x in s` rất nhanh.

In [3]:
s = set()
for x in [1, 2, 2, 1, 3, 3, 3]:
    s.add(x)
print(s, len(s))                  # {1, 2, 3} 3
print(2 in s, 9 in s)             # True False
print(type({}), type(set()))      # <class 'dict'> <class 'set'>

{1, 2, 3} 3
True False
<class 'dict'> <class 'set'>


### 3.3. `dict` mà mỗi giá trị là một `set`

Ta muốn **chia học sinh theo ghế**: khóa = số ghế, giá trị = tập học sinh ngồi ghế đó. Cách viết tay: nếu ghế chưa có trong `dict` thì tạo `set()` mới, rồi `.add(student_id)`. Với `collections.defaultdict(set)`, việc tạo `set()` rỗng cho khóa mới được làm **tự động** nên chỉ cần `d[bench_id].add(student_id)`.

In [4]:
# cách viết tay
d = {}
for student_id, bench_id in [[1, 2], [2, 2], [3, 3], [1, 3]]:
    if bench_id not in d:
        d[bench_id] = set()
    d[bench_id].add(student_id)
print(d)                           # {2: {1, 2}, 3: {1, 3}}

# cách dùng defaultdict(set)
d2 = defaultdict(set)
for student_id, bench_id in [[1, 2], [2, 2], [3, 3], [1, 3]]:
    d2[bench_id].add(student_id)
print(dict(d2), d2 == d)           # {2: {1, 2}, 3: {1, 3}} True

{2: {1, 2}, 3: {1, 3}}
{2: {1, 2}, 3: {1, 3}} True


### 3.4. `max(...)` trên các kích thước và tham số `default`

`max(map(len, d.values()))` lấy độ dài (số học sinh) của từng `set` rồi chọn lớn nhất. Nhưng `max` của một dãy **rỗng** báo `ValueError`. Khi `students = []`, `d` rỗng nên cần `default=0`: `max(..., default=0)`.

In [5]:
sizes = [2, 3]
print(max(sizes))                         # 3
try:
    max([])
except ValueError as e:
    print("ValueError:", e)
print(max([], default=0))                 # 0

3
ValueError: max() iterable argument is empty
0


### 3.5. `tuple` để đưa cặp vào `set`

Một `list` như `[1, 2]` **không** "băm" (hash) được, nên **không** thể là phần tử của `set` hay khóa của `dict`. `tuple` `(1, 2)` thì được. Vì vậy muốn loại trùng cả cặp `[student_id, bench_id]` ta đổi từng cặp thành `tuple`: `set(map(tuple, students))`.

In [6]:
students = [[1, 1], [1, 1], [2, 1]]
try:
    set(students)
except TypeError as e:
    print("TypeError:", e)             # unhashable type: 'list'
unique_pairs = set(map(tuple, students))
print(unique_pairs, len(unique_pairs))   # {(1, 1), (2, 1)} 2

TypeError: unhashable type: 'list'
{(1, 1), (2, 1)} 2


### 3.6. `sorted` và `itertools.groupby`

`sorted(...)` sắp xếp; với các `tuple` thì so sánh phần tử đầu rồi phần tử thứ hai. `itertools.groupby(iterable, key)` gom **các phần tử liên tiếp có cùng `key`** thành từng nhóm — vì vậy phải **sắp xếp trước theo đúng `key`** thì các phần tử cùng nhóm mới đứng liền nhau.

In [7]:
pairs = sorted([(2, 1), (1, 3), (1, 1), (3, 3)])     # sắp theo (ghế, học sinh)
print(pairs)
for bench, group in groupby(pairs, key=lambda p: p[0]):
    print("ghế", bench, "->", [p[1] for p in group])

[(1, 1), (1, 3), (2, 1), (3, 3)]
ghế 1 -> [1, 3]
ghế 2 -> [1]
ghế 3 -> [3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy ví dụ 1: `[[1,2],[2,2],[3,3],[1,3],[2,3]]`. Đi lần lượt từng dòng và **ghi vào ghế tương ứng**:

| dòng | `[học sinh, ghế]` | ghế 2 sau dòng này | ghế 3 sau dòng này |
|---|---|---|---|
| 1 | `[1, 2]` | `{1}` | `{}` |
| 2 | `[2, 2]` | `{1, 2}` | `{}` |
| 3 | `[3, 3]` | `{1, 2}` | `{3}` |
| 4 | `[1, 3]` | `{1, 2}` | `{3, 1}` |
| 5 | `[2, 3]` | `{1, 2}` | `{3, 1, 2}` |

Cuối cùng: ghế `2` có `2` học sinh, ghế `3` có `3` học sinh ⇒ đáp án `3`.

Còn nếu **chỉ đếm số dòng** của mỗi ghế: ghế `2` có `2` dòng, ghế `3` có `3` dòng — trùng hợp đúng ở ví dụ này, nhưng sai ở ví dụ 3 (`[[1,1],[1,1]]` có `2` dòng mà chỉ `1` học sinh). Vì thế phải dùng **tập hợp** để loại trùng.

In [8]:
students = [[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]
bench2, bench3 = set(), set()
for student_id, bench_id in students:
    (bench2 if bench_id == 2 else bench3).add(student_id)
    print(f"sau dòng [{student_id}, {bench_id}]: ghế 2 = {sorted(bench2)}, ghế 3 = {sorted(bench3)}")
print("đáp án =", max(len(bench2), len(bench3)))

sau dòng [1, 2]: ghế 2 = [1], ghế 3 = []
sau dòng [2, 2]: ghế 2 = [1, 2], ghế 3 = []
sau dòng [3, 3]: ghế 2 = [1, 2], ghế 3 = [3]
sau dòng [1, 3]: ghế 2 = [1, 2], ghế 3 = [1, 3]
sau dòng [2, 3]: ghế 2 = [1, 2], ghế 3 = [1, 2, 3]
đáp án = 3


## 5. Cách 1 — Brute Force / baseline: với mỗi ghế, quét lại cả danh sách

**Ý tưởng.** Bước 1: tìm danh sách các **ghế khác nhau** (duyệt và gom vào một `set`). Bước 2: với **mỗi ghế** `b`, duyệt lại **toàn bộ** `students`, mỗi khi gặp `[sid, b]` mà `sid` **chưa có** trong danh sách học sinh của ghế `b` thì thêm vào. Đếm độ dài danh sách, giữ giá trị lớn nhất.

Ở đây ta cố tình dùng `list` và phép kiểm tra `not in` (tốn `O(độ dài list)`) để thấy rõ chi phí khi chưa dùng `set`.

In [9]:
def max_students_brute(students: List[List[int]]) -> int:
    benches = set()
    for student_id, bench_id in students:
        benches.add(bench_id)               # các ghế xuất hiện trong dữ liệu
    best = 0
    for b in benches:
        seen = []                           # học sinh khác nhau trên ghế b
        for student_id, bench_id in students:
            if bench_id == b and student_id not in seen:
                seen.append(student_id)
        best = max(best, len(seen))
    return best


for students, expected in examples:
    print(max_students_brute(students), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(students)` và `B` là số ghế khác nhau. Mỗi ghế quét `n` dòng, và `not in seen` có thể tốn tới `O(n)` ⇒ **Time `O(B · n²)` trong trường hợp xấu nhất**, **Space `O(n)`**. Với `n <= 100` vẫn chạy nhanh, nhưng ta lặp đi lặp lại cùng một dữ liệu.

**Nút thắt:** mỗi ghế quét lại **toàn bộ** danh sách. Ta chỉ cần quét **một lần** và bỏ mỗi học sinh vào đúng "ngăn" của ghế mình; `set` thay `list` để loại trùng trong `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Gom nhóm một lượt.** Duyệt `students` một lần. Với mỗi `[student_id, bench_id]`, bỏ `student_id` vào **`set` của ghế `bench_id`** trong một `dict` (`ghế → set`). `set` tự bỏ qua học sinh trùng. Sau đó đáp án là **kích thước lớn nhất** của các `set`.

- Một lượt duyệt ⇒ `O(n)`.
- `set.add` và `in` trung bình `O(1)`.
- Xử lý riêng trường hợp rỗng (`default=0`).

Ví dụ: ví dụ 3 `[[1,1],[1,1]]` ⇒ `d = {1: {1}}` ⇒ kích thước `1`.

In [10]:
d = defaultdict(set)
for student_id, bench_id in [[1, 1], [1, 1]]:
    d[bench_id].add(student_id)
    print(f"[{student_id}, {bench_id}] -> d = {dict(d)}")
print("đáp án =", max(map(len, d.values()), default=0))

[1, 1] -> d = {1: {1}}
[1, 1] -> d = {1: {1}}
đáp án = 1


## 7. Các cách tiếp theo

### Cách 2 — `defaultdict(set)`: ghế → tập học sinh

**Giải thích từng đoạn.**

1. `d = defaultdict(set)`: từ điển mà khóa mới tự có `set()` rỗng.
2. `for student_id, bench_id in students: d[bench_id].add(student_id)`: bỏ học sinh vào ghế; trùng thì `set` tự bỏ qua.
3. `max(map(len, d.values()), default=0)`: `map(len, ...)` đổi mỗi `set` thành kích thước; `default=0` xử lý trường hợp không có học sinh.

In [11]:
def max_students_dict_of_sets(students: List[List[int]]) -> int:
    d = defaultdict(set)                          # ghế -> tập học sinh
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return max(map(len, d.values()), default=0)   # kích thước lớn nhất; rỗng -> 0


for students, expected in examples:
    print(max_students_dict_of_sets(students), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionDictOfSets:
    def maxStudentsOnBench(self, students: List[List[int]]) -> int:
        d = defaultdict(set)
        for student_id, bench_id in students:
            d[bench_id].add(student_id)
        return max(map(len, d.values()), default=0)


print(SolutionDictOfSets().maxStudentsOnBench([[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]))   # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (tối đa `n` học sinh được lưu trong các `set`).

### Cách 3 — Loại trùng cả cặp bằng `set`, rồi đếm theo ghế bằng `Counter`

**Giải thích từng đoạn.**

1. `pairs = set(map(tuple, students))`: mỗi cặp `[sid, bid]` thành `tuple` rồi bỏ vào `set` ⇒ mỗi **cặp (học sinh, ghế)** chỉ còn **một** bản.
2. Khi đã không còn cặp trùng, **số dòng của mỗi ghế = số học sinh khác nhau** trên ghế đó. Vậy `Counter(bench for _, bench in pairs)` đếm đúng.
3. Trả `max(counter.values(), default=0)`.

In [13]:
def max_students_dedup_pairs(students: List[List[int]]) -> int:
    pairs = set(map(tuple, students))                      # loại trùng cặp (học sinh, ghế)
    count = Counter(bench_id for _, bench_id in pairs)     # số học sinh khác nhau mỗi ghế
    return max(count.values(), default=0)


for students, expected in examples:
    print(max_students_dedup_pairs(students), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 4 — Sắp xếp cặp rồi `groupby` theo ghế

**Giải thích từng đoạn.**

1. Đổi mỗi cặp thành `(bench_id, student_id)` (ghế trước) và dùng `set` để loại trùng.
2. `sorted(...)`: các cặp cùng ghế đứng **liền nhau**.
3. `groupby(..., key=lambda p: p[0])` gom theo ghế; vì không còn cặp trùng, **độ dài của mỗi nhóm = số học sinh khác nhau**.
4. Lấy `max` độ dài nhóm. `sum(1 for _ in group)` đếm số phần tử của nhóm (nhóm là iterator nên không dùng `len` được).

In [14]:
def max_students_groupby(students: List[List[int]]) -> int:
    pairs = sorted({(bench_id, student_id) for student_id, bench_id in students})    # (ghế, học sinh), đã loại trùng và sắp xếp
    best = 0
    for bench_id, group in groupby(pairs, key=lambda p: p[0]):
        best = max(best, sum(1 for _ in group))
    return best


for students, expected in examples:
    print(max_students_groupby(students), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (do sắp xếp), **Space `O(n)`**. Chậm hơn Cách 2/3 một chút về lý thuyết nhưng không cần `dict`.

### Cách 5 — Bảng đánh dấu hai chiều (nhờ ràng buộc `1..100`)

**Giải thích từng đoạn.**

1. Vì `student_id` và `bench_id` đều trong `1..100`, ta tạo bảng `seen[bench][student]` kích thước `101 x 101` toàn `False` (bỏ hàng/cột `0`).
2. Mỗi dòng `[sid, bid]`: nếu `seen[bid][sid]` còn `False` thì đánh dấu `True` và tăng `count[bid]` thêm `1`; nếu đã `True` thì bỏ qua (trùng).
3. Đáp án là `max(count)` (mảng `count` luôn có phần tử, ban đầu toàn `0`, nên rỗng cho ra `0` không cần `default`).

In [15]:
def max_students_table(students: List[List[int]]) -> int:
    MAX_ID = 100
    seen = [[False] * (MAX_ID + 1) for _ in range(MAX_ID + 1)]     # seen[ghế][học sinh]
    count = [0] * (MAX_ID + 1)                                     # count[ghế] = số học sinh khác nhau
    for student_id, bench_id in students:
        if not seen[bench_id][student_id]:
            seen[bench_id][student_id] = True
            count[bench_id] += 1
    return max(count)


for students, expected in examples:
    print(max_students_table(students), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(n + M²)`** để dựng bảng `M x M` với `M = 100` (hằng số), **Space `O(M²)`**. Cách này chỉ hợp khi ID bị chặn nhỏ như ở đề; tổng quát nên dùng `dict`/`set`.

Chú ý: bảng tạo bằng `[[False] * 101 for _ in range(101)]` (mỗi hàng là một list **riêng**). Viết `[[False] * 101] * 101` sẽ khiến mọi hàng là **cùng một list** — một lỗi kinh điển.

## 8. TRACE từng bước

Ta trace Cách 2 (`dict` ghế → `set`) trên ví dụ 1 và ví dụ 3 (có cặp trùng), in ra `dict` sau mỗi dòng.

In [16]:
def trace_dict_of_sets(students):
    d = defaultdict(set)
    print("students =", students)
    for i, (student_id, bench_id) in enumerate(students):
        before = len(d[bench_id])
        d[bench_id].add(student_id)
        after = len(d[bench_id])
        note = "mới" if after > before else "TRÙNG, bỏ qua"
        print(f"  dòng {i}: [{student_id}, {bench_id}] ({note}) -> {{{', '.join(f'{b}: {sorted(v)}' for b, v in sorted(d.items()))}}}")
    sizes = {b: len(v) for b, v in sorted(d.items())}
    print("  kích thước từng ghế:", sizes)
    ans = max(sizes.values(), default=0)
    print("  đáp án =", ans)
    return ans


assert trace_dict_of_sets([[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]) == 3
assert trace_dict_of_sets([[1, 1], [1, 1]]) == 1
assert trace_dict_of_sets([]) == 0

students = [[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]
  dòng 0: [1, 2] (mới) -> {2: [1]}
  dòng 1: [2, 2] (mới) -> {2: [1, 2]}
  dòng 2: [3, 3] (mới) -> {2: [1, 2], 3: [3]}
  dòng 3: [1, 3] (mới) -> {2: [1, 2], 3: [1, 3]}
  dòng 4: [2, 3] (mới) -> {2: [1, 2], 3: [1, 2, 3]}
  kích thước từng ghế: {2: 2, 3: 3}
  đáp án = 3
students = [[1, 1], [1, 1]]
  dòng 0: [1, 1] (mới) -> {1: [1]}
  dòng 1: [1, 1] (TRÙNG, bỏ qua) -> {1: [1]}
  kích thước từng ghế: {1: 1}
  đáp án = 1
students = []
  kích thước từng ghế: {}
  đáp án = 0


In [17]:
def trace_table(students):
    MAX_ID = 100
    seen = [[False] * (MAX_ID + 1) for _ in range(MAX_ID + 1)]
    count = [0] * (MAX_ID + 1)
    print("students =", students)
    for i, (sid, bid) in enumerate(students):
        if seen[bid][sid]:
            print(f"  dòng {i}: [{sid}, {bid}] đã đánh dấu -> bỏ qua")
        else:
            seen[bid][sid] = True
            count[bid] += 1
            print(f"  dòng {i}: [{sid}, {bid}] đánh dấu mới -> count[{bid}] = {count[bid]}")
    ans = max(count)
    print("  đáp án =", ans)
    return ans


assert trace_table([[1, 1], [2, 1], [3, 1], [4, 2], [5, 2]]) == 3
assert trace_table([[1, 1], [1, 1]]) == 1

students = [[1, 1], [2, 1], [3, 1], [4, 2], [5, 2]]
  dòng 0: [1, 1] đánh dấu mới -> count[1] = 1
  dòng 1: [2, 1] đánh dấu mới -> count[1] = 2
  dòng 2: [3, 1] đánh dấu mới -> count[1] = 3
  dòng 3: [4, 2] đánh dấu mới -> count[2] = 1
  dòng 4: [5, 2] đánh dấu mới -> count[2] = 2
  đáp án = 3
students = [[1, 1], [1, 1]]
  dòng 0: [1, 1] đánh dấu mới -> count[1] = 1
  dòng 1: [1, 1] đã đánh dấu -> bỏ qua
  đáp án = 1


Bảng trace Cách 2 với `students = [[1,2],[2,2],[3,3],[1,3],[2,3]]`:

| dòng | `[sid, bid]` | `d` sau dòng này | ghi chú |
|---|---|---|---|
| 0 | `[1, 2]` | `{2: {1}}` | mới |
| 1 | `[2, 2]` | `{2: {1, 2}}` | mới |
| 2 | `[3, 3]` | `{2: {1, 2}, 3: {3}}` | mới |
| 3 | `[1, 3]` | `{2: {1, 2}, 3: {1, 3}}` | mới |
| 4 | `[2, 3]` | `{2: {1, 2}, 3: {1, 2, 3}}` | mới |

Kích thước: ghế `2` là `2`, ghế `3` là `3` ⇒ đáp án `3`.

## 9. Code nộp LeetCode

Mình nộp Cách 2 (`defaultdict(set)`): ngắn, `O(n)`, xử lý được danh sách rỗng. Trên LeetCode, `List` và `defaultdict` thường đã có sẵn; để chạy ở máy bạn, ghi rõ dòng `import`.

In [18]:
from collections import defaultdict
from typing import List


class Solution:
    def maxStudentsOnBench(self, students: List[List[int]]) -> int:
        d = defaultdict(set)                          # ghế -> tập học sinh khác nhau
        for student_id, bench_id in students:
            d[bench_id].add(student_id)
        return max(map(len, d.values()), default=0)   # danh sách rỗng -> 0


sol = Solution()
for students, expected in examples:
    print(students, "->", sol.maxStudentsOnBench(students), "(kỳ vọng", expected, ")")

[[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]] -> 3 (kỳ vọng 3 )
[[1, 1], [2, 1], [3, 1], [4, 2], [5, 2]] -> 3 (kỳ vọng 3 )
[[1, 1], [1, 1]] -> 1 (kỳ vọng 1 )
[] -> 0 (kỳ vọng 0 )


## 10. Vì sao thuật toán đúng

- **Mỗi ghế có một `set` riêng:** khóa của `dict` là `bench_id`, nên học sinh của ghế này không lẫn sang ghế khác (một học sinh có thể thuộc `set` của nhiều ghế — đúng yêu cầu vì mỗi ghế đếm riêng).
- **Loại trùng đúng chỗ:** `set.add` bỏ qua học sinh đã có **trong cùng ghế**. Vì vậy học sinh lặp lại trên cùng ghế chỉ tính một lần, đúng ghi chú của đề.
- **Đếm đủ:** mỗi dòng `[sid, bid]` đều được duyệt và thêm vào `set` của `bid`, nên cuối cùng `set` của ghế chứa **đúng** các học sinh khác nhau ngồi ở ghế đó.
- **Cực đại:** đáp án là `max` kích thước các `set`; không ghế nào thiếu vì mọi ghế có mặt trong dữ liệu đều có khóa trong `dict`.
- **Trường hợp rỗng:** `d` rỗng ⇒ `max(..., default=0)` trả `0`, đúng đề.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(students)`, `B` là số ghế khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Mỗi ghế quét lại cả danh sách (list + `not in`) | `O(B · n²)` xấu nhất | `O(n)` |
| 2. `defaultdict(set)` | `O(n)` | `O(n)` |
| 3. `set` các cặp + `Counter` | `O(n)` | `O(n)` |
| 4. Sắp xếp + `groupby` | `O(n log n)` | `O(n)` |
| 5. Bảng đánh dấu `101 x 101` | `O(n + M²)`, `M = 100` | `O(M²)` |

Với `n <= 100` mọi cách đều tức thì; Cách 2 hoặc 3 là gọn nhất và vẫn đúng khi ID lớn hơn nhiều.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng bằng `assert`.

In [19]:
sol = Solution()
impls = {
    "brute": max_students_brute,
    "dict_of_sets": max_students_dict_of_sets,
    "dedup_pairs": max_students_dedup_pairs,
    "groupby": max_students_groupby,
    "table": max_students_table,
    "leetcode": sol.maxStudentsOnBench,
}


def check_all(students, expected, label):
    for key, fn in impls.items():
        actual = fn([row[:] for row in students])      # truyền bản sao để chắc chắn không ai sửa input
        assert actual == expected, (label, key, students, actual, expected)
    print("PASS", label)


for idx, (students, expected) in enumerate(examples, 1):
    check_all(students, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
PASS ví dụ 4


### 12.1. Các trường hợp biên

In [20]:
check_all([], 0, "rỗng")
check_all([[5, 7]], 1, "một học sinh")
check_all([[1, 1]] * 10, 1, "một học sinh lặp 10 lần")
check_all([[i, 1] for i in range(1, 101)], 100, "100 học sinh khác nhau trên một ghế")
check_all([[i, i] for i in range(1, 101)], 1, "mỗi học sinh một ghế riêng")
check_all([[1, b] for b in range(1, 101)], 1, "một học sinh ngồi 100 ghế khác nhau")
check_all([[1, 1], [2, 1], [1, 2], [2, 2]], 2, "hai ghế bằng nhau")
check_all([[1, 1], [1, 1], [2, 2], [3, 2]], 2, "ghế 1 lặp nên chỉ 1, ghế 2 có 2")
check_all([[100, 100], [99, 100]], 2, "ID biên 100")
check_all([[1, 1], [2, 2], [2, 2], [2, 2], [3, 3], [3, 3]], 1, "toàn lặp")

PASS rỗng
PASS một học sinh
PASS một học sinh lặp 10 lần
PASS 100 học sinh khác nhau trên một ghế
PASS mỗi học sinh một ghế riêng
PASS một học sinh ngồi 100 ghế khác nhau
PASS hai ghế bằng nhau
PASS ghế 1 lặp nên chỉ 1, ghế 2 có 2
PASS ID biên 100
PASS toàn lặp


### 12.2. Phản ví dụ cho các cách làm sai

Hai cách sai điển hình: (a) **đếm số dòng** của mỗi ghế (không loại trùng); (b) ánh xạ **học sinh → ghế** thay vì **ghế → học sinh**.

In [21]:
def wrong_count_rows(students):
    count = Counter(bench_id for _, bench_id in students)      # SAI: đếm số dòng, không loại trùng
    return max(count.values(), default=0)


def wrong_reverse_map(students):
    d = {}
    for student_id, bench_id in students:
        d[student_id] = bench_id                                # SAI: một học sinh chỉ giữ được một ghế
    return max(Counter(d.values()).values(), default=0)


print(wrong_count_rows([[1, 1], [1, 1]]), "!=", max_students_dict_of_sets([[1, 1], [1, 1]]))
print(wrong_reverse_map([[1, 1], [1, 2], [1, 3], [2, 1]]), "!=", max_students_dict_of_sets([[1, 1], [1, 2], [1, 3], [2, 1]]))
assert wrong_count_rows([[1, 1], [1, 1]]) != 1
assert wrong_reverse_map([[1, 1], [1, 2], [1, 3], [2, 1]]) != 2

2 != 1
1 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh `students` ngẫu nhiên **đúng ràng buộc**: số dòng `0..100`, mỗi `student_id`, `bench_id` trong `1..100`. Để có nhiều cặp trùng ta thường chọn tập ID nhỏ (ví dụ chỉ `1..5`). Mọi cách phải cho cùng kết quả với Cách 1.

In [22]:
def valid(students):
    return (0 <= len(students) <= 100
            and all(len(row) == 2 and 1 <= row[0] <= 100 and 1 <= row[1] <= 100 for row in students))


rng = random.Random(3450)
for trial in range(3000):
    n = rng.choice([0, 1, 2, 5, 10, 30, 100])
    hi_s = rng.choice([1, 2, 3, 5, 10, 100])
    hi_b = rng.choice([1, 2, 3, 5, 10, 100])
    students = [[rng.randint(1, hi_s), rng.randint(1, hi_b)] for _ in range(n)]
    assert valid(students)
    ref = max_students_brute(students)
    for key, fn in impls.items():
        assert fn([row[:] for row in students]) == ref, (key, students)
print("PASS 3000 input ngẫu nhiên đúng ràng buộc")

PASS 3000 input ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số dòng của mỗi ghế (không loại trùng)

In [23]:
def sai_dem_dong(students):
    count = {}
    for student_id, bench_id in students:
        count[bench_id] = count.get(bench_id, 0) + 1       # SAI: học sinh lặp bị đếm nhiều lần
    return max(count.values(), default=0)


print("sai:", sai_dem_dong([[1, 1], [1, 1]]), "| đúng:", Solution().maxStudentsOnBench([[1, 1], [1, 1]]))

sai: 2 | đúng: 1


**Vì sao sai:** đề nói học sinh lặp lại trên cùng ghế chỉ tính **một lần**. Đếm số dòng cho `2` ở ví dụ 3, trong khi đáp án đúng là `1`. **Cách sửa:** dùng `set` cho mỗi ghế (hoặc loại trùng cặp trước).

### Lỗi 2: `max` của dãy rỗng

In [24]:
def sai_max_rong(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return max(map(len, d.values()))                      # SAI: students = [] thì max() báo lỗi


try:
    sai_max_rong([])
except ValueError as e:
    print("ValueError:", e)
print("đúng:", Solution().maxStudentsOnBench([]))

ValueError: max() iterable argument is empty
đúng: 0


**Vì sao sai:** khi `students = []`, `d` rỗng nên `max(...)` không có phần tử nào để so sánh và báo `ValueError`. Đề yêu cầu trả `0`. **Cách sửa:** `max(..., default=0)` hoặc `if not students: return 0`.

### Lỗi 3: Ánh xạ ngược `học sinh → ghế`

In [25]:
def sai_anh_xa_nguoc(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[student_id].add(bench_id)                       # SAI: khóa là học sinh
    return max(map(len, d.values()), default=0)


data = [[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]
print("sai:", sai_anh_xa_nguoc(data), "| đúng:", Solution().maxStudentsOnBench(data))

sai: 2 | đúng: 3


**Vì sao sai:** đảo vai trò cho ra "một học sinh ngồi nhiều ghế nhất" (ở đây học sinh `1` và `2` đều ngồi `2` ghế ⇒ `2`), không phải "ghế có nhiều học sinh nhất" (`3`). **Cách sửa:** khóa là **ghế**, giá trị là **tập học sinh**. Đọc kỹ cặp là `[student_id, bench_id]` (học sinh trước, ghế sau).

### Lỗi 4: Dùng `list` thay `set` và quên loại trùng

In [26]:
def sai_dung_list(students):
    d = defaultdict(list)
    for student_id, bench_id in students:
        d[bench_id].append(student_id)                    # SAI: list không loại trùng
    return max(map(len, d.values()), default=0)


print("sai:", sai_dung_list([[1, 1], [1, 1], [2, 2]]), "| đúng:", Solution().maxStudentsOnBench([[1, 1], [1, 1], [2, 2]]))

sai: 2 | đúng: 1


**Vì sao sai:** `list.append` luôn thêm phần tử, kể cả trùng. Dữ liệu `[[1,1],[1,1],[2,2]]` cho `2` thay vì `1`. **Cách sửa:** dùng `set` và `.add`.

### Lỗi 5: Trả về mã ghế thay vì số học sinh

In [27]:
def sai_tra_ma_ghe(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return max(d, key=lambda b: len(d[b]), default=0)      # SAI: trả về KHÓA (mã ghế)


data = [[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]
print("sai:", sai_tra_ma_ghe(data), "| đúng:", Solution().maxStudentsOnBench(data))

sai: 3 | đúng: 3


**Vì sao sai:** `max(d, key=...)` trả về **khóa** (mã ghế `3`) chứ không phải số học sinh. Ở ví dụ này tình cờ cùng bằng `3` nên dễ lọt; thử ghế mang mã khác (ví dụ ghế `7` có 2 học sinh) sẽ thấy ngay sự khác biệt. **Cách sửa:** `max(map(len, d.values()), default=0)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Mỗi nhóm có bao nhiêu phần tử khác nhau?**" ⇒ `dict` nhóm → `set`; hoặc loại trùng cặp trước rồi đếm bằng `Counter`.
- "Đếm **duy nhất** / **distinct** / **unique**" ⇒ nghĩ tới `set`.
- Nhớ phân biệt: `set` để **loại trùng**, `Counter` để **đếm số lần**, `defaultdict(list)` để **giữ danh sách** (có trùng).
- Cặp `[a, b]` muốn đưa vào `set`/khóa `dict` phải đổi thành `tuple`.
- Luôn nghĩ tới **input rỗng**: `max(..., default=0)`.
- Đọc kỹ thứ tự trong cặp: `[student_id, bench_id]` (đừng đảo).
- Khi không chắc đề (bài Premium), viết thêm các ca tự dựng vào test và đối chiếu lại đề gốc.

## 15. Biến thể của bài

1. **Mã ghế đông nhất:** trả về `bench_id` có nhiều học sinh khác nhau nhất (hòa thì lấy mã nhỏ nhất).
2. **Tổng số học sinh khác nhau** trên **toàn bộ** các ghế.
3. **Số ghế có ít nhất `2` học sinh**, hoặc ghế có **ít** học sinh nhất.
4. **Học sinh ngồi nhiều ghế:** liệt kê các `student_id` xuất hiện ở từ `2` ghế trở lên.
5. Bài cùng họ: **Contains Duplicate (217)**, **Intersection of Two Arrays (349)**, **Unique Number of Occurrences (1207)**.

In [28]:
def busiest_bench(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    if not d:
        return None
    return min(d, key=lambda b: (-len(d[b]), b))             # nhiều học sinh nhất, hòa thì mã nhỏ


def total_unique_students(students):
    return len({student_id for student_id, _ in students})


def benches_with_at_least_two(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return sum(1 for v in d.values() if len(v) >= 2)


def students_on_many_benches(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[student_id].add(bench_id)
    return sorted(sid for sid, benches in d.items() if len(benches) >= 2)


data = [[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]
print(busiest_bench(data), total_unique_students(data), benches_with_at_least_two(data), students_on_many_benches(data))
assert busiest_bench(data) == 3
assert busiest_bench([[1, 5], [2, 5], [3, 4], [4, 4]]) == 4         # hòa 2-2, chọn mã nhỏ
assert busiest_bench([]) is None
assert total_unique_students(data) == 3
assert benches_with_at_least_two(data) == 2
assert students_on_many_benches(data) == [1, 2]

3 3 2 [1, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mỗi ghế quét lại danh sách | list + `not in` | `O(B · n²)` | `O(n)` | hiểu đề, đối chiếu |
| 2. `defaultdict(set)` | ghế → tập học sinh | `O(n)` | `O(n)` | **nên nộp** |
| 3. `set` cặp + `Counter` | loại trùng cặp rồi đếm theo ghế | `O(n)` | `O(n)` | gọn, hai bước rõ ràng |
| 4. Sắp xếp + `groupby` | các cặp cùng ghế đứng liền nhau | `O(n log n)` | `O(n)` | khi muốn dữ liệu đã sắp xếp |
| 5. Bảng `101 x 101` | đánh dấu `seen[ghế][học sinh]` | `O(n + M²)` | `O(M²)` | ID bị chặn nhỏ như đề |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `len({1, 2, 2, 3})`, `max([], default=0)`, `set(map(tuple, [[1, 1], [1, 1]]))`, `Solution().maxStudentsOnBench([[1, 1], [2, 1], [2, 1]])`.

**Bài 2 (tính tay).** Với `students = [[1,5],[2,5],[1,5],[3,4],[4,4],[5,4],[3,4]]`: lập bảng ghế → tập học sinh khác nhau rồi tính đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_students_todo(students)` theo Cách 2.

In [29]:
def max_students_todo(students):
    # TODO 1: tạo d = defaultdict(set)  (ghế -> tập học sinh)
    # TODO 2: duyệt từng [student_id, bench_id]: thêm student_id vào d[bench_id]
    # TODO 3: trả về kích thước lớn nhất của các set (nhớ trường hợp rỗng -> 0)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_students_counter2(students)` theo Cách 3 **nhưng không dùng `Counter`**: sau khi có `pairs = set(map(tuple, students))`, tự đếm bằng `dict` (`cnt[bench] = cnt.get(bench, 0) + 1`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `bench_sizes(students)` trả về `dict` `ghế -> số học sinh khác nhau`, sắp theo mã ghế tăng dần. (b) Viết `second_largest_bench_size(students)` trả về **kích thước lớn thứ hai** trong các ghế (tính theo số học sinh khác nhau; nếu chỉ có `0` hoặc `1` ghế thì trả `0`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `3`, `0`, `{(1, 1)}`, `2`. Bài 3: `defaultdict(set)`, `d[bench_id].add(student_id)`, `max(map(len, d.values()), default=0)`.
- **Mức 2 (vừa):** Bài 2: ghế `5` có `{1, 2}` (2 học sinh; dòng `[1,5]` thứ hai trùng), ghế `4` có `{3, 4, 5}` (3 học sinh; `[3,4]` thứ hai trùng) ⇒ đáp án `3`. Bài 4: đếm `for _, bench in pairs`.
- **Mức 3 (sâu):** Bài 5(a): `{b: len(v) for b, v in sorted(d.items())}`. 5(b): sắp xếp các kích thước giảm dần (`sorted(..., reverse=True)`) rồi lấy phần tử thứ hai (vị trí `1`); hai ghế bằng nhau vẫn tính là lớn nhất và lớn thứ hai. Nhớ kiểm tra có ít nhất hai ghế.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert len({1, 2, 2, 3}) == 3
assert max([], default=0) == 0
assert set(map(tuple, [[1, 1], [1, 1]])) == {(1, 1)}
assert Solution().maxStudentsOnBench([[1, 1], [2, 1], [2, 1]]) == 2

# Bài 2
data2 = [[1, 5], [2, 5], [1, 5], [3, 4], [4, 4], [5, 4], [3, 4]]
assert Solution().maxStudentsOnBench(data2) == 3


# Bài 3
def max_students_todo_ans(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return max(map(len, d.values()), default=0)


for students, expected in examples:
    assert max_students_todo_ans(students) == expected


# Bài 4
def max_students_counter2(students):
    pairs = set(map(tuple, students))
    cnt = {}
    for _, bench in pairs:
        cnt[bench] = cnt.get(bench, 0) + 1
    return max(cnt.values(), default=0)


for students, expected in examples:
    assert max_students_counter2(students) == expected


# Bài 5(a)
def bench_sizes(students):
    d = defaultdict(set)
    for student_id, bench_id in students:
        d[bench_id].add(student_id)
    return {b: len(v) for b, v in sorted(d.items())}


assert bench_sizes([[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]) == {2: 2, 3: 3}
assert bench_sizes([]) == {}
assert list(bench_sizes([[1, 9], [1, 3]])) == [3, 9]


# Bài 5(b)
def second_largest_bench_size(students):
    sizes = sorted(bench_sizes(students).values(), reverse=True)
    return sizes[1] if len(sizes) >= 2 else 0


assert second_largest_bench_size([[1, 2], [2, 2], [3, 3], [1, 3], [2, 3]]) == 2
assert second_largest_bench_size([[1, 1], [2, 1]]) == 0
assert second_largest_bench_size([]) == 0
assert second_largest_bench_size([[1, 1], [2, 1], [3, 2], [4, 2]]) == 2
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gom nhóm theo ghế, loại trùng bằng `set`, lấy kích thước lớn nhất)

```python
from collections import defaultdict
from typing import List


class Solution:
    def maxStudentsOnBench(self, students: List[List[int]]) -> int:
        d = defaultdict(set)
        for student_id, bench_id in students:
            d[bench_id].add(student_id)
        return max(map(len, d.values()), default=0)
```

**Checklist**

- [ ] Khóa của `dict` là **ghế** (`bench_id`), giá trị là **tập học sinh**?
- [ ] Dùng `set` (loại trùng), không dùng `list`?
- [ ] Trả về **số học sinh** (kích thước), không phải mã ghế?
- [ ] Trường hợp `students = []` trả `0` (`default=0`)?
- [ ] Đã thử `[[1,1],[1,1]]` (`1`) và `[[1,2],[2,2],[3,3],[1,3],[2,3]]` (`3`)?
- [ ] Skeleton đúng `class Solution` / `def maxStudentsOnBench(self, students: List[List[int]]) -> int`?
- [ ] Bài Premium: đã đối chiếu lại đề gốc (nếu có tài khoản)?